In [ ]:
# !gdown --fuzzy https://drive.google.com/file/d/1jgys2TpiZ-BJhNwB-axepvkw0Cc0mYTb/view?usp=sharing
# !gdown --fuzzy https://drive.google.com/file/d/1ZXJaZ5WwijaS0HOxLo6T5K4Mo_JQzOPL/view?usp=sharing
# !gdown --fuzzy https://drive.google.com/file/d/13OVqyoAB4P4bdvsQP8Y3_YX16ntGLrLZ/view?usp=sharing
# !unzip -q fonts.zip
# !unzip -q background.zip
# !unzip -q dl_4.zip
# !rm dl_4.zip fonts.zip background.zip

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance, ImageOps
from tqdm import tqdm
import albumentations as A
import cv2

from transformers import (
    VisionEncoderDecoderModel, TrOCRProcessor,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
)

from sklearn.model_selection import train_test_split

dataset_path = '/lab/dl_4/'

train_dir = os.path.join(dataset_path, 'train/train')
test_dir = os.path.join(dataset_path, 'test/test')
train_df = pd.read_csv(os.path.join(dataset_path, 'train.csv'))

avg_background = np.load('/lab/avg_background.npy')
orange_background = np.load('/lab/orange_background.npy')
white_background = np.load('/lab/white_background.npy')

device = "cuda" if torch.cuda.is_available() else "cpu"

/home/ubuntu/.jupytervenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def clean_text(text):
    text = ''.join([c for c in str(text) if c.isdigit()])
    return text if text != "" else "0"


In [ ]:
def generate_syntetic(num_len):
  backgrounds = [
      avg_background,
      white_background,
      orange_background,
  ]
  back_idx = np.random.choice(np.arange(0, len(backgrounds)), p=np.array([0.75, 0.1, 0.15]))
  IMAGE = Image.fromarray(backgrounds[back_idx])
  IMAGE_SIZE = IMAGE.size
  font_size = np.random.uniform(0.32, 0.5) * IMAGE_SIZE[0]
  font_path = np.random.choice(['/lab/GoogleSans-Bold.ttf', '/lab/Korolev-Bold.otf', '/lab/Lato-Black.ttf'])
  font = ImageFont.truetype(font_path, size = font_size)
  draw = ImageDraw.Draw(IMAGE)
  text = str(np.random.randint(0, 10 ** (num_len)))

  bbox = draw.textbbox((0, 0), text, font = font)
  text_w, text_h = bbox[2] - bbox[0], bbox[3] - bbox[1]
  x = (IMAGE_SIZE[0] - text_w)//2 + np.random.randint(-2, 4)
  y = (IMAGE_SIZE[1] - text_h)//2 + np.random.randint(-5, 4)

  rand_fill = tuple(np.random.randint(0, 60, 3))
  draw.text((x,y), text, font=font, fill=rand_fill)


  if random.random() < 0.7:
    IMAGE = IMAGE.rotate(random.uniform(-15, 15), fillcolor=(32, 46, 22))

  if random.random() < 0.6:
    arr = np.array(IMAGE)
    noise = np.random.normal(0,  10, arr.shape).astype(np.uint8)
    dst_poisson = np.random.poisson(arr / 255 * 100) / 100 * 255
    arr = np.clip(dst_poisson, 0, 255).astype(np.uint8)
    arr = np.clip(arr + noise, 0, 255)
    IMAGE = Image.fromarray(arr)


  if random.random() < 0.5:
    IMAGE = IMAGE.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1)))
  if random.random() < 0.5:
    IMAGE = ImageEnhance.Contrast(IMAGE).enhance(np.random.uniform(0.5, 1.5))
  if random.random() < 0.7:
    brightness_factor = np.random.uniform(0.4, 1.4)
    arr = np.array(IMAGE)
    arr = np.clip(arr * brightness_factor, 0, 255).astype(np.uint8)
    IMAGE = Image.fromarray(arr)


  if np.random.random() < 0.5:
    img_array = np.array(IMAGE).astype(np.float32)
    h, w = img_array.shape[:2]
    x = np.linspace(-1, 1, w)
    y = np.linspace(-1, 1, h)
    xx, yy = np.meshgrid(x, y)
    angle = np.random.uniform(0, 2 * np.pi)
    gradient = (xx * np.cos(angle) + yy * np.sin(angle))
    gradient = (gradient - gradient.min()) / (gradient.max() - gradient.min())
    gradient = gradient * np.random.uniform(0.3, 0.8) + np.random.uniform(0.3, 0.7)

    img_array = np.clip(img_array * gradient[:, :, np.newaxis], 0, 255)
    IMAGE = Image.fromarray(img_array.astype(np.uint8))

  if np.random.random() < 0.5:
    import io
    buffer = io.BytesIO()
    IMAGE.save(buffer, format='JPEG', quality = np.random.randint(30, 80))
    buffer.seek(0)
    IMAGE = Image.open(buffer)

  if random.random() < 0.4:
    arr = np.array(IMAGE)
    h, w = arr.shape[:2]
    src = np.float32([[0,0],[w,0],[w,h],[0,h]])
    dx, dy = w*0.05, h*0.08
    dst = np.float32([
        [np.random.uniform(0, dx), np.random.uniform(0, dy)],
        [w - np.random.uniform(0, dx), np.random.uniform(0, dy)],
        [w - np.random.uniform(0, dx), h - np.random.uniform(0, dy)],
        [np.random.uniform(0, dx), h - np.random.uniform(0, dy)],
    ])
    M = cv2.getPerspectiveTransform(src, dst)
    arr = cv2.warpPerspective(arr, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    IMAGE = Image.fromarray(arr)

  if np.random.random() < 0.6:
      IMAGE = ImageOps.autocontrast(IMAGE)
  # plt.axis('off')
  # plt.imshow(IMAGE)
  # plt.show()
  return IMAGE, int(text)

for i in range(25):
  syn, t = generate_syntetic(4)
  print(t)

4077
3457
2033
7756
4747
6247
1806
3970
9219
9476
2178
5856
7564
3613
8337
7074
1548
6363
4692
9477
2445
5461
3943
8013
4005


In [ ]:
class OCRDataset(torch.utils.data.Dataset):
    def __init__(self, df, root_dir, processor, transform = None, synthetic_ratio=0.3):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.processor = processor
        self.synthetic_ratio = synthetic_ratio
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        # # --- synthetic branch ---
        if random.random() < self.synthetic_ratio:
            num_len = np.random.choice(np.arange(1, 5), p=(0.15, 0.35, 0.40, 0.1))
            image, price = generate_syntetic(num_len)
            price = clean_text(price)

        else:
            filename = self.df.iloc[idx]['Filename']
            price = clean_text(self.df.iloc[idx]['Price'])

            #image = Image.open(os.path.join(self.root_dir, filename)).convert("RGB")
            image = cv2.imread(os.path.join(self.root_dir, filename))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            if self.transform:
                augmented = self.transform(image = image)
                image = augmented['image']
            image = Image.fromarray(image)
            # autocontrast
        image = ImageOps.exif_transpose(image)

        #image = self.augment(image)

        image = ImageOps.pad(image, (384, 384), method = Image.BICUBIC)

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        labels = self.processor.tokenizer(
            price,
            padding="max_length",
            truncation=True,
            max_length=8,
            return_tensors="pt"
        ).input_ids.squeeze()

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
aug = A.Compose([
    A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, fill=(255,255,255), p=0.3),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.6),
    A.GaussianBlur(blur_limit=(3, 5), p=0.45),
    A.Sharpen(p=0.2),
    A.Illumination(p=0.5),
    A.GaussNoise(std_range=(0.1, 0.2), p=0.4),
    A.CoarseDropout(num_holes_range=(1,2), hole_height_range=(2,4), hole_width_range=(2,4), p=0.15),
])

In [ ]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")

model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.use_cache = False

Loading weights: 100%|███████████████████████| 478/478 [00:02<00:00, 176.85it/s]
[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)

train_dataset = OCRDataset(train_df, train_dir, processor, transform = aug, synthetic_ratio=0.65)
val_dataset = OCRDataset(val_df, train_dir, processor, synthetic_ratio=0.0)

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, processor.tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)

    pred_texts = processor.batch_decode(preds, skip_special_tokens=True)
    label_texts = processor.batch_decode(labels, skip_special_tokens=True)

    pred_texts = [clean_text(p) for p in pred_texts]
    label_texts = [clean_text(t) for t in label_texts]

    acc = np.mean([p == t for p, t in zip(pred_texts, label_texts)])
    return {"accuracy": acc}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/lab/DL_4_models/large",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,

    num_train_epochs=18,
    learning_rate=3e-5,

    lr_scheduler_type="cosine",
    warmup_steps=1000,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    predict_with_generate=True,
    generation_num_beams = 5,
    generation_max_length = 6,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    save_total_limit=2,

    fp16=True,
    logging_steps=100,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=10)]
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 0}.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.192224,0.042100,0.975415
2,0.155073,0.017841,0.986711
3,0.116706,0.015166,0.992027
4,0.102113,0.036176,0.974086
5,0.079199,0.022392,0.987375
6,0.068498,0.019684,0.989369
7,0.077988,0.013307,0.991362
8,0.064335,0.013473,0.994020
9,0.063183,0.010671,0.993355
10,0.042530,0.009411,0.995349


Writing model shards: 100%|███████████████████████| 1/1 [00:01<00:00,  1.93s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].


TrainOutput(global_step=15246, training_loss=0.10210516974945699, metrics={'train_runtime': 7675.972, 'train_samples_per_second': 31.763, 'train_steps_per_second': 1.986, 'total_flos': 1.8243941241672696e+20, 'train_loss': 0.10210516974945699, 'epoch': 18.0})

In [ ]:
model.save_pretrained("/lab/DL_4_models/large/last")
processor.save_pretrained("/lab/DL_4_models/large/last")

Writing model shards: 100%|███████████████████████| 1/1 [00:07<00:00,  7.23s/it]


['/lab/DL_4_models/large/last/processor_config.json']

In [ ]:
model.eval()

submission = pd.read_csv(os.path.join(dataset_path, 'sample_submission.csv'))
pred_labels = []

def preprocess(image):
    image = ImageOps.exif_transpose(image)
    image = ImageOps.pad(image, (384, 384), method=Image.BICUBIC)
    return image

def predict_with_tta(image):
    transforms = [
        lambda x: x,
        lambda x: x.rotate(-15, fillcolor=(255,255,255)),
        lambda x: x.rotate(15, fillcolor=(255,255,255)),
        lambda x: ImageEnhance.Contrast(x).enhance(1.3),
    ]

    preds = []

    for t in transforms:
        img = t(image.copy())
        img = preprocess(img)

        pixel_values = processor(images=img, return_tensors="pt").pixel_values.to(device)

        generated_ids = model.generate(
            pixel_values,
            max_length=6,
            num_beams=5,
            early_stopping=True
        )

        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        text = clean_text(text)

        if text.isdigit() and 1 <= len(text) <= 4:
            preds.append(text)

    # if len(preds) == 0:
    #     return ""

    return max(set(preds), key=preds.count)


for filename in tqdm(submission["Filename"]):
    image = Image.open(os.path.join(test_dir, filename)).convert("RGB")
    pred = predict_with_tta(image)
    pred_labels.append(pred)

100%|███████████████████████████████████████| 3762/3762 [21:22<00:00,  2.93it/s]


In [ ]:
submission["Price"] = pred_labels
submission.to_csv("/lab/submission_trOCR.csv", index=False)

print("\n submission_trOCR.csv saved!")


 submission_trOCR.csv saved!


In [ ]:
#Mainland-Black
#Glot Round Bold